In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

In [22]:
INSTALLED__PAYMENTS_DATA_PATH = "../datasets/raw/installments_payments.csv"

BERAU_DATA_PATH = "../datasets/raw/bureau.csv"

PREVIOUS_APPLICATION_DATA_PATH = "../datasets/raw/previous_application.csv"
IMAGES_PATH = "../images/"
OUTPUT_PATH = "../datasets/preprocess/"
TRAIN_DATA_PATH = "../datasets/raw/application_train.csv"

In [23]:
bureau  = pd.read_csv(BERAU_DATA_PATH)

In [3]:
app_train = pd.read_csv(TRAIN_DATA_PATH)

previous_app = pd.read_csv(PREVIOUS_APPLICATION_DATA_PATH)

installments = pd.read_csv(INSTALLED__PAYMENTS_DATA_PATH)

display(installments.head())
print("\n")
print(installments.info())

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 8 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_PREV              int64  
 1   SK_ID_CURR              int64  
 2   NUM_INSTALMENT_VERSION  float64
 3   NUM_INSTALMENT_NUMBER   int64  
 4   DAYS_INSTALMENT         float64
 5   DAYS_ENTRY_PAYMENT      float64
 6   AMT_INSTALMENT          float64
 7   AMT_PAYMENT             float64
dtypes: float64(5), int64(3)
memory usage: 830.4 MB
None


In [13]:
installments['DAYS_LATE'] = installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']
installments['AMT_SHORTFALL'] = installments['AMT_INSTALMENT'] - installments['AMT_PAYMENT']

installments.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,DAYS_LATE,AMT_SHORTFALL
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360,-7.0,0.000
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525,0.0,0.000
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000,0.0,0.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130,-8.0,0.000
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585,17.0,4.455


In [19]:
installments_agg = installments.groupby('SK_ID_PREV').agg({
    'AMT_INSTALMENT':'count',
    'DAYS_LATE': ['mean','max'],
    'AMT_SHORTFALL':['mean','max']
})


installments_agg.columns = ['_'.join(col).upper() for col in installments_agg.columns]
installments_agg = installments_agg.reset_index()

installments_agg.head()

,SK_ID_PREV,AMT_INSTALMENT_COUNT,DAYS_LATE_MEAN,DAYS_LATE_MAX,AMT_SHORTFALL_MEAN,AMT_SHORTFALL_MAX
0,1000001,2,-16.000000,-6.0,0.000000,0.000
1,1000002,4,-19.750000,-5.0,0.000000,0.000
2,1000003,3,-15.333333,-14.0,0.000000,0.000
3,1000004,7,-26.714286,-10.0,0.000000,0.000
4,1000005,11,-8.454545,3.0,1337.600455,14710.815


In [20]:
#merge previous app data with installments data
previous_app = previous_app.merge(installments_agg, on='SK_ID_PREV', how='left')

previous_app.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,AMT_INSTALMENT_count,DAYS_LATE_mean,DAYS_LATE_max,AMT_SHORTFALL_mean,AMT_SHORTFALL_max,AMT_INSTALMENT_COUNT,DAYS_LATE_MEAN,DAYS_LATE_MAX,AMT_SHORTFALL_MEAN,AMT_SHORTFALL_MAX
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,1.0,0.000000,0.0,0.0,0.0,1.0,0.000000,0.0,0.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,5.0,-9.200000,-7.0,0.0,0.0,5.0,-9.200000,-7.0,0.0,0.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,9.0,-8.222222,1.0,0.0,0.0,9.0,-8.222222,1.0,0.0,0.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,11.0,-7.090909,0.0,0.0,0.0,11.0,-7.090909,0.0,0.0,0.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
previous_app_agg = previous_app.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'AMT_CREDIT': ['mean', 'max', 'sum'],
    'AMT_ANNUITY': ['mean', 'max'],
    'CNT_PAYMENT': ['mean', 'max'],
    'DAYS_DECISION': ['mean', 'min'],
    'DAYS_LATE_MEAN': 'mean',           # promedio del atraso promedio de cada solicitud
    'DAYS_LATE_MAX': 'max',             # el peor atraso de todas sus solicitudes
    'AMT_SHORTFALL_MEAN': 'mean',
    'AMT_SHORTFALL_MAX': 'max',
})

previous_app_agg.columns = ['_'.join(col).upper() for col in previous_app_agg.columns]
previous_app_agg = previous_app_agg.reset_index()

display(previous_app_agg.head())

,SK_ID_CURR,SK_ID_PREV_COUNT,AMT_CREDIT_MEAN,AMT_CREDIT_MAX,AMT_CREDIT_SUM,AMT_ANNUITY_MEAN,AMT_ANNUITY_MAX,CNT_PAYMENT_MEAN,CNT_PAYMENT_MAX,DAYS_DECISION_MEAN,DAYS_DECISION_MIN,DAYS_LATE_MEAN_MEAN,DAYS_LATE_MAX_MAX,AMT_SHORTFALL_MEAN_MEAN,AMT_SHORTFALL_MAX_MAX
0,100001,1,23787.00,23787.0,23787.0,3951.000,3951.000,8.0,8.0,-1740.0,-1740,-15.500000,-6.0,0.0,0.0
1,100002,1,179055.00,179055.0,179055.0,9251.775,9251.775,24.0,24.0,-606.0,-606,-20.421053,-12.0,0.0,0.0
2,100003,3,484191.00,1035882.0,1452573.0,56553.990,98356.995,10.0,12.0,-1305.0,-2341,-7.448413,-1.0,0.0,0.0
3,100004,1,20106.00,20106.0,20106.0,5357.250,5357.250,4.0,4.0,-815.0,-815,-7.666667,-3.0,0.0,0.0
4,100005,2,20076.75,40153.5,40153.5,4813.200,4813.200,12.0,12.0,-536.0,-757,-23.555556,1.0,0.0,0.0
